In [224]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import Tuple,List
import numpy as np
from dataclasses import dataclass
import os
from pathlib import Path
from tqdm import tqdm
from PIL import Image
from torch.utils.data import Dataset,DataLoader
import torchvision.transforms as transforms
from torchvision.utils import save_image
import matplotlib.pyplot as plt

In [225]:
@dataclass
class DiTConfig:
  hidden_size:int=768
  depth:int=12
  num_heads:int=12
  mlp_ratio:float=4.0
  patch_size:int=2
  in_channels:int=3
  latent_channels:int=4
  num_timesteps:int=1000
  beta_start:float=0.0001
  beta_end:float=0.02
  lr:float=1e-4
  weight_decay:float=0.0
  grad_clip:float=1.0
  image_size:int=256
  batch_size:int=8
  num_workers:int=2
  num_epochs:int=100
  save_every:int=10
  device:str = 'cuda' if torch.cuda.is_available() else 'cpu'
  mixed_precision:bool=True
  data_path:str = "./data/CelebA-HQ-img"
  checkpoint_path:str="./checkpoints"
  sample_path:str='./samples'

  def __post_init__(self):
    os.makedirs(self.checkpoint_path,exist_ok=True)
    os.makedirs(self.sample_path,exist_ok=True)

In [226]:
import kagglehub
from pathlib import Path

DOWNLOAD_ROOT = Path(DiTConfig.data_path)
DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)

# Public Kaggle dataset
download_path = kagglehub.dataset_download(
    "ipythonx/celebamaskhq",
    output_dir=str(DOWNLOAD_ROOT),
)

print("Downloaded to:", download_path)

Using Colab cache for faster access to the 'celebamaskhq' dataset.
Downloaded to: /kaggle/input/celebamaskhq


In [227]:
class ResnetBlock(nn.Module):
  def __init__(self, in_channels: int, out_channels: int):
    super().__init__()
    num_groups1 = min(32, in_channels)
    self.norm1 = nn.GroupNorm(num_groups1, in_channels)
    self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)

    num_groups2 = min(32, out_channels)
    self.norm2 = nn.GroupNorm(num_groups2, out_channels)
    self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)

    self.shortcut = nn.Identity() if in_channels == out_channels else nn.Conv2d(in_channels, out_channels, 1)

  def forward(self, x):
    h = self.norm1(x)
    h = F.silu(h)
    h = self.conv1(h)
    h = self.norm2(h) # FIX: Apply norm2 to the output of conv1, which is 'h'
    h = F.silu(h)
    h = self.conv2(h)
    return h + self.shortcut(x)

In [228]:
class Encoder(nn.Module):
  def __init__(self,in_channels:int=3,latent_channels:int=4,
               base_channels:int=128,channel_mults=(1,2,4,4)):
    super().__init__()
    self.conv_in = nn.Conv2d(in_channels,base_channels,3,padding=1)
    self.down_blocks = nn.ModuleList()
    in_ch = base_channels
    for mult in channel_mults:
      block = nn.ModuleList()
      out_ch = base_channels * mult
      block.append(ResnetBlock(in_ch,out_ch))
      block.append(ResnetBlock(out_ch,out_ch))
      if mult != channel_mults[-1]:
        block.append(nn.Conv2d(out_ch,out_ch,3,stride=2,padding=1))
      self.down_blocks.append(block)
      in_ch = out_ch
    self.mid = nn.ModuleList([ResnetBlock(in_ch,in_ch),ResnetBlock(in_ch,in_ch)])
    # Fix: Ensure num_groups is divisible by in_ch
    self.norm_out = nn.GroupNorm(min(32, in_ch),in_ch)
    self.conv_out = nn.Conv2d(in_ch,2*latent_channels,3,padding=1)

  def forward(self,x):
    h = self.conv_in(x)
    for block in self.down_blocks:
      for layer in block:
        h = layer(h)

    for layer in self.mid:
      h = layer(h)

    h = self.norm_out(h)
    h = F.silu(h)
    h = self.conv_out(h)
    mean, logvar = h.chunk(2,dim=1)
    return mean,logvar

In [229]:
class Decoder(nn.Module):
  def __init__(self,latent_channels:int=4,out_channels:int=3,
               base_channels:int=128,channel_mults=(1,2,4,4)):
    super().__init__()
    in_ch = base_channels * channel_mults[-1]
    self.mid = nn.ModuleList([ResnetBlock(in_ch,in_ch),ResnetBlock(in_ch,in_ch)])
    self.up_blocks = nn.ModuleList()
    for i,mult in reversed(list(enumerate(channel_mults))):
      out_ch = base_channels * mult
      block = nn.ModuleList()
      block.append(ResnetBlock(in_ch,out_ch))
      block.append(ResnetBlock(out_ch,out_ch))
      block.append(ResnetBlock(out_ch,out_ch))

      if i!=0:
        block.append(nn.Upsample(scale_factor=2,mode='nearest'))
      self.up_blocks.append(block)
      in_ch = out_ch
    # Fix: Ensure num_groups is divisible by in_ch
    self.norm_out = nn.GroupNorm(min(32, in_ch),in_ch)
    self.conv_out = nn.Conv2d(in_ch,out_channels,3,padding=1)

  def forward(self,x):
    h = x
    for layer in self.mid:
      h = layer(h)

    for block in self.up_blocks:
      for layer in block:
        h = layer(h)

    h = self.norm_out(h)
    h = F.silu(h)
    h = self.conv_out(h)
    return h

In [230]:
class VAE(nn.Module):
  def __init__(self,latent_channels:int=4,base_channels:int=128):
    super().__init__()
    self.encoder = Encoder(latent_channels=latent_channels,base_channels=base_channels)
    self.decoder = Decoder(latent_channels=latent_channels,base_channels=base_channels)
    self.latent_channels = latent_channels

  def encode(self,x:torch.Tensor)->torch.Tensor:
    mean,logvar = self.encoder(x)
    return mean*0.18215

  def decode(self,z:torch.Tensor)-> torch.Tensor:
    z = z/0.18215
    return self.decoder(z)

  def forward(self,x:torch.Tensor):
    mean,logvar = self.encoder(x)
    std = torch.exp(0.5*logvar)
    eps = torch.randn_like(mean)
    z = mean + eps * std
    kl = -0.5 * torch.mean(1+logvar-mean.pow(2)-logvar.exp())
    x_recon = self.decode(z)
    return x_recon,kl,z

In [231]:
class Encoder(nn.Module):
  def __init__(self, in_channels: int = 3, latent_channels: int = 4,
               base_channels: int = 128, channel_mults=(1, 2, 4)):
    super().__init__()
    self.conv_in = nn.Conv2d(in_channels, base_channels, 3, padding=1)
    self.down_blocks = nn.ModuleList()
    in_ch = base_channels
    for i, mult in enumerate(channel_mults):
      block = nn.ModuleList()
      out_ch = base_channels * mult
      block.append(ResnetBlock(in_ch, out_ch))
      block.append(ResnetBlock(out_ch, out_ch))
      if i != len(channel_mults) - 1:
        block.append(nn.Conv2d(out_ch, out_ch, 3, stride=2, padding=1))
      self.down_blocks.append(block)
      in_ch = out_ch

    self.mid = nn.ModuleList([
        ResnetBlock(in_ch, in_ch),
        ResnetBlock(in_ch, in_ch)
    ])

    self.norm_out = nn.GroupNorm(min(32, in_ch), in_ch)
    self.conv_out = nn.Conv2d(in_ch, 2 * latent_channels, 3, padding=1)

  def forward(self, x):
    h = self.conv_in(x)
    for block in self.down_blocks:
      for layer in block:
        h = layer(h)
    for layer in self.mid:
      h = layer(h)
    h = self.norm_out(h)
    h = F.silu(h)
    h = self.conv_out(h)
    mean, logvar = h.chunk(2, dim=1)
    return mean, logvar

In [232]:
class Decoder(nn.Module):
  def __init__(self, latent_channels: int = 4, out_channels: int = 3,
               base_channels: int = 128, channel_mults=(1, 2, 4)):
    super().__init__()
    current_ch = base_channels * channel_mults[-1]
    self.conv_in_latent = nn.Conv2d(latent_channels, current_ch, 3, padding=1)

    self.mid = nn.ModuleList([
        ResnetBlock(current_ch, current_ch),
        ResnetBlock(current_ch, current_ch)
    ])

    self.up_blocks = nn.ModuleList()
    for i, mult in reversed(list(enumerate(channel_mults))):
      out_ch = base_channels * mult
      block = nn.ModuleList()
      block.append(ResnetBlock(current_ch, out_ch))
      block.append(ResnetBlock(out_ch, out_ch))
      block.append(ResnetBlock(out_ch, out_ch))
      if i != 0:
        block.append(nn.Upsample(scale_factor=2, mode='nearest'))
      self.up_blocks.append(block)
      current_ch = out_ch

    self.norm_out = nn.GroupNorm(min(32, current_ch), current_ch)
    self.conv_out = nn.Conv2d(current_ch, out_channels, 3, padding=1)

  def forward(self, x):
    h = self.conv_in_latent(x)
    for layer in self.mid:
      h = layer(h)
    for block in self.up_blocks:
      for layer in block:
        h = layer(h)
    h = self.norm_out(h)
    h = F.silu(h)
    h = self.conv_out(h)
    return h

In [233]:
class VAE(nn.Module):
  def __init__(self, latent_channels: int = 4, base_channels: int = 128):
    super().__init__()
    # Re-initializing with updated Encoder/Decoder classes
    self.encoder = Encoder(latent_channels=latent_channels, base_channels=base_channels)
    self.decoder = Decoder(latent_channels=latent_channels, base_channels=base_channels)
    self.latent_channels = latent_channels

  def encode(self, x: torch.Tensor) -> torch.Tensor:
    mean, logvar = self.encoder(x)
    return mean * 0.18215

  def decode(self, z: torch.Tensor) -> torch.Tensor:
    z = z / 0.18215
    return self.decoder(z)

  def forward(self, x: torch.Tensor):
    mean, logvar = self.encoder(x)
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(mean)
    z = mean + eps * std
    kl = -0.5 * torch.mean(1 + logvar - mean.pow(2) - logvar.exp())
    x_recon = self.decode(z)
    return x_recon, kl, z

In [234]:
class PatchEmbed(nn.Module):
  def __init__(self,image_size=256,patch_size=2,in_channels=4,hidden_size=768):
    super().__init__()
    self.image_size = image_size
    self.patch_size = patch_size
    self.num_patches = (image_size // patch_size) ** 2
    self.proj = nn.Conv2d(in_channels, hidden_size, kernel_size=patch_size, stride=patch_size)

  def forward(self, x):
    x = self.proj(x)
    x = x.flatten(2).transpose(1, 2)
    return x

In [235]:
class TimestepEmbedder(nn.Module):
  def __init__(self,hidden_size:int,frequency_embedding_size:int=256):
    super().__init__()
    self.mlp = nn.Sequential(
        nn.Linear(frequency_embedding_size,hidden_size),
        nn.SiLU(),
        nn.Linear(hidden_size,hidden_size)
    )
    self.frequency_embedding_size = frequency_embedding_size

  @staticmethod
  def timestep_embedding(t:torch.Tensor,dim:int,max_period:int=10000):
    half = dim//2
    freqs = torch.exp(
        -math.log(max_period) * torch.arange(start=0,end=half,dtype=torch.float32) / half
    ).to(device=t.device)

    args = t[:,None].float() * freqs[None]
    embedding = torch.cat([torch.cos(args),torch.sin(args)],dim=-1)

    if dim % 2==1:
      embedding = torch.cat([embedding,torch.zeros_like(embedding[:,:1])],dim=-1)
    return embedding

  def forward(self,t:torch.Tensor) -> torch.Tensor:
    t_freq = self.timestep_embedding(t,self.frequency_embedding_size)
    return self.mlp(t_freq)

In [236]:
class Attention(nn.Module):
  def __init__(self,hidden_size:int,num_heads:int=12,qkv_bias:bool=False):
    super().__init__()
    self.num_heads = num_heads
    self.head_dim = hidden_size//num_heads
    self.qkv = nn.Linear(hidden_size,hidden_size*3,bias=qkv_bias)
    self.proj = nn.Linear(hidden_size,hidden_size)

  def forward(self,x:torch.Tensor) -> torch.Tensor:
    B,N,C = x.shape
    qkv = self.qkv(x).reshape(B,N,3,self.num_heads,self.head_dim).permute(2,0,3,1,4)
    q,k,v = qkv[0], qkv[1], qkv[2]
    attn = (q @ k.transpose(-2,-1))*(self.head_dim** -0.5)
    attn = F.softmax(attn,dim=-1)
    x = (attn @ v).transpose(1,2).reshape(B,N,C)
    x = self.proj(x)
    return x

In [237]:
class Mlp(nn.Module):
  def __init__(self,hidden_size:int,mlp_ratio:float=4.0):
    super().__init__()
    inner = int(hidden_size * mlp_ratio)
    self.fc1 = nn.Linear(hidden_size,inner)
    self.fc2 = nn.Linear(inner,hidden_size)
    self.act = nn.GELU(approximate='tanh')

  def forward(self,x:torch.Tensor):
    x = self.fc1(x)
    x = self.act(x)
    x = self.fc2(x)
    return x

In [238]:
class DiTBlock(nn.Module):
  def __init__(self,hidden_size:int,num_heads:int,mlp_ratio:float=4.0):
    super().__init__()
    self.norm1 = nn.LayerNorm(hidden_size,elementwise_affine=False,eps=1e-6)
    self.attn = Attention(hidden_size,num_heads)
    self.norm2 = nn.LayerNorm(hidden_size,elementwise_affine=False,eps=1e-6)
    self.mlp = Mlp(hidden_size,mlp_ratio)
    self.adaLN = nn.Sequential(
        nn.SiLU(),
        nn.Linear(hidden_size,6*hidden_size)
    )
    nn.init.zeros_(self.adaLN[1].weight)
    nn.init.zeros_(self.adaLN[1].bias)

  def forward(self,x:torch.Tensor,c:torch.Tensor)->torch.Tensor:
    shift_msa,scale_msa,gate_msa,shift_mlp,scale_mlp,gate_mlp = self.adaLN(c).chunk(6,dim=1)
    shift_msa = shift_msa.unsqueeze(1)
    scale_msa = scale_msa.unsqueeze(1)
    gate_msa = gate_msa.unsqueeze(1)
    shift_mlp = shift_mlp.unsqueeze(1)
    scale_mlp = scale_mlp.unsqueeze(1)
    gate_mlp = gate_mlp.unsqueeze(1)
    h = self.norm1(x)
    h = h * (1+scale_msa) + shift_msa
    h = self.attn(h)
    x = x + gate_msa * h
    h = self.norm2(x)
    h = h * (1+scale_mlp) + shift_mlp
    h = self.mlp(h)
    x = x + gate_mlp * h
    return x

In [239]:
class FinalLayer(nn.Module):
  def __init__(self,hidden_size:int,patch_size:int,out_channels:int):
    super().__init__()
    self.norm = nn.LayerNorm(hidden_size,elementwise_affine=False,eps=1e-6)
    self.proj = nn.Linear(hidden_size,patch_size**2*out_channels)
    self.adaLN = nn.Sequential(
        nn.SiLU(),
        nn.Linear(hidden_size,2*hidden_size)
    )
    nn.init.zeros_(self.adaLN[1].weight)
    nn.init.zeros_(self.adaLN[1].bias)

  def forward(self,x:torch.Tensor,c:torch.Tensor)->torch.Tensor:
    shift,scale = self.adaLN(c).chunk(2,dim=1)
    shift = shift.unsqueeze(1)
    scale = scale.unsqueeze(1)
    h = self.norm(x)
    h = h * (1+scale) + shift
    h = self.proj(h)
    return h

In [240]:
class DiT(nn.Module):
  def __init__(self, config: DiTConfig):
    super().__init__()
    self.config = config
    # Use the image_size from config (which is the latent size, e.g., 16)
    self.patch_embed = PatchEmbed(
        image_size=config.image_size,
        patch_size=config.patch_size,
        in_channels=config.in_channels,
        hidden_size=config.hidden_size
    )
    num_patches = self.patch_embed.num_patches
    # Force initialization to the correct number of patches (e.g., 64)
    self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, config.hidden_size))
    self.t_embedder = TimestepEmbedder(config.hidden_size)
    self.blocks = nn.ModuleList([
        DiTBlock(config.hidden_size, config.num_heads, config.mlp_ratio) for _ in range(config.depth)
    ])

    self.final = FinalLayer(config.hidden_size, config.patch_size, config.latent_channels)
    self.initialize_weights()

  def initialize_weights(self):
    nn.init.trunc_normal_(self.pos_embed, std=0.02)
    for block in self.blocks:
      nn.init.xavier_uniform_(block.attn.qkv.weight, gain=0.02)
      nn.init.xavier_uniform_(block.attn.proj.weight, gain=0.02)
      nn.init.xavier_uniform_(block.mlp.fc1.weight, gain=0.02)
      nn.init.xavier_uniform_(block.mlp.fc2.weight, gain=0.02)
    nn.init.zeros_(self.final.proj.weight)
    nn.init.zeros_(self.final.proj.bias)

  def unpatchify(self, x: torch.Tensor) -> torch.Tensor:
    B = x.shape[0]
    H = W = self.config.image_size
    P = self.config.patch_size
    C = self.config.latent_channels
    x = x.reshape(B, H // P, W // P, P, P, C)
    x = x.permute(0, 5, 1, 3, 2, 4).reshape(B, C, H, W)
    return x

  def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    x = self.patch_embed(x)
    x = x + self.pos_embed
    t_emb = self.t_embedder(t)
    for block in self.blocks:
      x = block(x, t_emb)
    x = self.final(x, t_emb)
    x = self.unpatchify(x)
    return x

  def trainable_params(self):
    return sum(p.numel() for p in self.parameters() if p.requires_grad)

In [241]:
class DiffusionScheduler:
  def __init__(self,config:DiTConfig):
    self.num_timesteps = config.num_timesteps
    # Ensure all tensors are created on the specified device
    device = config.device
    betas = torch.linspace(config.beta_start,config.beta_end,config.num_timesteps,dtype=torch.float32, device=device)
    alphas = 1-betas
    alphas_cumprod = torch.cumprod(alphas,dim=0)
    self.betas = betas
    self.alphas_cumprod = alphas_cumprod
    self.sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
    self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1-alphas_cumprod)

  def _get_values(self,buffer:torch.Tensor,t:torch.Tensor,shape)->torch.Tensor:
    return buffer[t].reshape(-1,*([1]*(len(shape)-1))).to(config.device)

  def add_noise(self,x0:torch.Tensor,t:torch.Tensor):
    noise = torch.randn_like(x0)
    sqrt_alphas = self._get_values(self.sqrt_alphas_cumprod,t,x0.shape)
    sqrt_one_minus = self._get_values(self.sqrt_one_minus_alphas_cumprod,t,x0.shape)
    xt = sqrt_alphas * x0 + sqrt_one_minus * noise
    return xt,noise

  def sample_timesteps(self,batch_size:int,device:torch.device):
    return torch.randint(0,self.num_timesteps,(batch_size,),device=device,dtype=torch.long)

In [242]:
class CelebA_HQ_Dataset(Dataset):
  def __init__(self,root_dir:str,image_size:int=256,limit:int=None):
    self.root_dir = Path(root_dir)
    self.image_paths = list(self.root_dir.glob("*.jpg"))+list(self.root_dir.glob("*.png"))
    if len(self.image_paths)==0:
      raise ValueError(f"No images found in {root_dir}")
    if limit is not None and limit < len(self.image_paths):
        self.image_paths = self.image_paths[:limit] # Limit the number of images

    self.transform = transforms.Compose([
        transforms.Resize((image_size,image_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
    ])

  def __len__(self):
    return len(self.image_paths)

  def __getitem__(self,idx:int):
    img = Image.open(self.image_paths[idx]).convert('RGB')
    return self.transform(img)

In [243]:
class DiTTrainer:
  def __init__(self,config:DiTConfig):
    self.config = config
    self.device = config.device
    self.vae = VAE(latent_channels=config.latent_channels).to(self.device)
    self.dit = DiT(config).to(self.device)
    self.scheduler = DiffusionScheduler(config)
    self.optimizer = torch.optim.AdamW(self.dit.parameters(),lr=config.lr,weight_decay=config.weight_decay)
    self.vae_optimizer = torch.optim.AdamW(self.vae.parameters(), lr=1e-4)
    self.scaler = torch.amp.GradScaler('cuda') if config.mixed_precision and torch.cuda.is_available() else None

  def train_vae(self, dataloader, epochs=5):
    self.vae.train()
    for epoch in range(epochs):
      pbar = tqdm(dataloader, desc=f"VAE Epoch {epoch+1}")
      for batch in pbar:
        batch = batch.to(self.device)
        recon, kl, _ = self.vae(batch)
        recon_loss = F.mse_loss(recon, batch)
        loss = recon_loss + 0.00001 * kl
        self.vae_optimizer.zero_grad()
        loss.backward()
        self.vae_optimizer.step()
        pbar.set_postfix({"recon_loss": f"{recon_loss.item():.4f}"})

  def train_step(self,images:torch.Tensor):
    images = images.to(self.device)
    with torch.no_grad():
      latents = self.vae.encode(images)
    t = self.scheduler.sample_timesteps(images.shape[0],self.device)
    noisy,noise = self.scheduler.add_noise(latents,t)
    if self.scaler is not None:
      with torch.amp.autocast('cuda'):
        noise_pred = self.dit(noisy,t)
        loss = F.mse_loss(noise_pred,noise)
      self.scaler.scale(loss).backward()
      self.scaler.unscale_(self.optimizer)
      torch.nn.utils.clip_grad_norm_(self.dit.parameters(),self.config.grad_clip)
      self.scaler.step(self.optimizer)
      self.scaler.update()
    else:
      noise_pred = self.dit(noisy,t)
      loss = F.mse_loss(noise_pred,noise)
      self.optimizer.zero_grad()
      loss.backward()
      torch.nn.utils.clip_grad_norm_(self.dit.parameters(),self.config.grad_clip)
      self.optimizer.step()
    self.optimizer.zero_grad()
    return loss

  def train(self,dataloader:DataLoader,num_epochs:int=None):
    print("Starting VAE Pre-training...")
    self.train_vae(dataloader, epochs=5)
    num_epochs = num_epochs or self.config.num_epochs
    self.dit.train()
    global_step=0
    for epoch in range(num_epochs):
      epoch_loss = 0.0
      pbar = tqdm(dataloader,desc=f"DiT Epoch {epoch+1}/{num_epochs}")
      for batch in pbar:
        loss = self.train_step(batch)
        epoch_loss += loss.item()
        global_step += 1
        pbar.set_postfix({"loss":f"{loss.item():.4f}"})
      if (epoch+1) % self.config.save_every ==0:
        self.save_checkpoint(epoch)
      print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss / len(dataloader):.4f}")

  @torch.no_grad()
  def sample(self,num_samples:int=8,steps:int=50)->torch.Tensor:
    self.dit.eval()
    # Image size here refers to latent spatial dimensions
    latent_size = self.config.image_size
    latents = torch.randn(num_samples, self.config.latent_channels, latent_size, latent_size, device=self.device)
    for i in tqdm(reversed(range(self.scheduler.num_timesteps)), total=self.scheduler.num_timesteps, desc="Sampling"): # Ensure tqdm for visibility
      t = torch.full((num_samples,), i, device=self.device, dtype=torch.long)
      noise_pred = self.dit(latents, t)

      # DDPM sampling step
      alpha_t = self.scheduler.alphas_cumprod[i]
      alpha_t_minus_1 = self.scheduler.alphas_cumprod[i-1] if i > 0 else torch.tensor(1.0, device=self.device)
      beta_t = self.scheduler.betas[i]

      # Calculate predicted x0
      x0_pred = (latents - torch.sqrt(1 - alpha_t) * noise_pred) / torch.sqrt(alpha_t)

      # Calculate mean for x_{t-1}
      mu = (torch.sqrt(alpha_t_minus_1) * beta_t * x0_pred + torch.sqrt(alpha_t) * (1 - alpha_t_minus_1) * latents) / (1 - alpha_t)

      variance = beta_t * (1 - alpha_t_minus_1) / (1 - alpha_t)

      if i > 0:
        noise = torch.randn_like(latents)
        latents = mu + torch.sqrt(variance) * noise
      else:
        latents = mu # No noise for the final step

    images = self.vae.decode(latents)
    return ((images + 1) / 2).clamp(0, 1)

  def sample_and_save(self,step:int):
    samples = self.sample(num_samples=8,steps=50)
    save_image(samples,os.path.join(self.config.sample_path,f"sample_step_{step}.png"),nrow=4)

  def save_checkpoint(self,epoch:int):
    ckpt = {"epoch": epoch, "dit_state_dict": self.dit.state_dict(), "vae_state_dict": self.vae.state_dict(), "optimizer_state_dict": self.optimizer.state_dict(), "config": self.config}
    path = os.path.join(self.config.checkpoint_path,f"dit_epoch_{epoch+1}.pt")
    torch.save(ckpt,path)
    print(f"Saved checkpoint: {path}")

In [244]:
class DiTInference:
  def __init__(self,checkpoint_path:str,device:str='cuda'):
    self.device = device
    # Fix: Set weights_only=False to allow loading of custom classes like DiTConfig
    ckpt = torch.load(checkpoint_path,map_location=device, weights_only=False)
    self.config = ckpt['config']
    # Uses the fixed VAE class
    self.vae = VAE(latent_channels=self.config.latent_channels).to(device)
    self.vae.load_state_dict(ckpt['vae_state_dict'])
    self.dit = DiT(self.config).to(self.device)
    self.dit.load_state_dict(ckpt['dit_state_dict'])
    self.scheduler = DiffusionScheduler(self.config)
    self.vae.eval()
    self.dit.eval()

  @torch.no_grad()
  def generate(self,num_samples:int=8,steps:int=250)->List[Image.Image]:
    latents = torch.randn(num_samples, self.config.latent_channels, self.config.image_size, self.config.image_size, device=self.device)
    T = self.scheduler.num_timesteps
    for t in tqdm(reversed(range(T)),total=min(steps,T),desc="Generating"):
      t_batch = torch.full((num_samples,), t, device=self.device, dtype=torch.long)
      noise_pred = self.dit(latents, t_batch)
      alpha = self.scheduler.alphas_cumprod[t]
      beta = self.scheduler.betas[t]
      latents = (latents - beta * noise_pred / (1 - alpha).sqrt()) / alpha.sqrt()
      if t > 0:
        latents = latents + torch.randn_like(latents) * beta.sqrt()
      # Defensive: Replace any NaN values in latents immediately
      if torch.isnan(latents).any():
          print(f"Warning: NaN encountered in latents at timestep {t}, replacing with 0.") # Corrected: t.item() removed
          latents = torch.nan_to_num(latents, nan=0.0)

    images = self.vae.decode(latents)
    images = (images + 1) / 2
    images = images.clamp(0,1)
    pil_images = []
    for img in images:
      # Replace any NaN values with 0.0 before converting to numpy
      if torch.isnan(img).any():
          print("Warning: NaN encountered in final image tensor, replacing with 0.")
          img = torch.nan_to_num(img, nan=0.0)
      arr = (img.permute(1,2,0).cpu().numpy()*255).astype(np.uint8)
      pil_images.append(Image.fromarray(arr))
    return pil_images

In [245]:
config = DiTConfig(
        hidden_size=512,
        depth=8,
        num_heads=8,
        patch_size=2,
        image_size=128,
        latent_channels=4,
        batch_size=2,
        num_epochs=10,
        lr=1e-4,
        data_path="data/CelebA-HQ-img/CelebAMask-HQ/CelebA-HQ-img",
        checkpoint_path="./checkpoints",
        sample_path="./samples",
        device="cuda" if torch.cuda.is_available() else "cpu",
        mixed_precision=True,
    )

config.in_channels = config.latent_channels
# Corrected: The VAE downsamples by 2^2 = 4 for the chosen channel_mults=(1,2,4).
# This means a 128x128 image becomes a 32x32 latent.
latent_size = config.image_size // 4

dit_model_config = DiTConfig(**config.__dict__)
dit_model_config.image_size = latent_size

print(f"Target Resolution: {config.image_size}")
print(f"Latent Space Size: {latent_size}")

# Prepare fresh dataset and loader
dataset = CelebA_HQ_Dataset(config.data_path, config.image_size, limit=100)
dataloader = DataLoader(dataset, batch_size=config.batch_size, shuffle=True, num_workers=2)

# Force complete re-initialization of the model state (trainer, VAE, and DiT)
trainer = DiTTrainer(dit_model_config)

# Start Training
trainer.train(dataloader, num_epochs=config.num_epochs)

# Inference
inference = DiTInference(f"./checkpoints/dit_epoch_{config.num_epochs}.pt", config.device)
images = inference.generate(num_samples=8, steps=50)

for i, img in enumerate(images):
    img.save(f"generated_{i}.png")

Target Resolution: 128
Latent Space Size: 32
Starting VAE Pre-training...


DiT Epoch 1/10: 100%|██████████| 50/50 [00:04<00:00, 11.72it/s, loss=0.1778]


Epoch 1/10, Loss: 0.5674


DiT Epoch 2/10: 100%|██████████| 50/50 [00:03<00:00, 12.54it/s, loss=0.0385]


Epoch 2/10, Loss: 0.1337


...

Generating: 424it [00:18, 24.02it/s]

Generating: 484it [00:20, 24.11it/s]

Generating: 565it [00:24, 23.94it/s]

Generating: 688it [00:29, 23.98it/s]

Generating: 1000it [00:42, 23.42it/s]
